# Chat27 — C1 official 100-epoch fresh/resume dual-T4 launch notebook (PRE-AUTHORIZATION)

**Purpose:** one reusable, guarded Kaggle Save Version / Save & Run All notebook, launching independent official MV00 and MV01 native sessions only after **separate explicit user authorization** and a private matching JSON approval record. By default `EXECUTE_APPROVED_OFFICIAL=False`: the notebook performs source/data/plan validation only, **NO GPU training**. The assistant creating this notebook did not authorize official training, and its publication does not authorize GPU execution.

**Scientific boundary:** immutable R9 C1 code release `fa6a7675daae385a04c7b77e69a9de29f9e1b2f3`; frozen science ancestor `b783eb3a470ab010785a8a24a90a666e9a3f2ed6`. MV00 SCConv-Early YOLO11s; MV01 same plus VGRA V1, both initialized FRESH from original SCConv-Early pretrained, NOT nonofficial Chat26 acceptance capsules. B-TRAIN/B-VAL only; B-TEST SEALED.

**Kaggle mounts:** choose accelerator **T4×2**, Internet enabled (fixed clone). Attach the exact original Chat26-successful Split-B and six fixed-source datasets, including the original R7D recovery config and corrected Chat25 R0 restager (either file directly or the exact Chat25 portable ZIP with SHA `3f0a386ace43453921ddd84b0e5f1aef5dacb72493d11fd5fbadcfad3c974e3c`). Preserve exact original asset hashes. Do not upload approval to the public GitHub repository.

**Before official execution:** independently review this notebook and its source against the pinned release, record your explicit consent, mount a **private** `CHAT27_C1_OFFICIAL_USER_AUTHORIZATION.json` matching the frozen C1 approval schema and run IDs, then change `EXECUTE_APPROVED_OFFICIAL=True` and use Kaggle **Save Version / Save & Run All** once in a fresh environment.

**For each RESUME session:** first download prior session's original per-arm G4 `.zipblob` and `.zipblob.sha256`, independently inspect on Windows using pinned R9 transport verifier, create the external attestation JSON for each arm, upload all to a private Kaggle dataset, and update **each arm's** `start`, `target`, `resume_sha256` below. Never resume from a local-only or nonofficial checkpoint. Different arms may stop at different epochs.

**Output integrity:** successful local output is still **OFFSITE_UNVERIFIED**. Save the Kaggle version, download both arm G4 archives and logs, independently verify bytes on Windows before any next session. If Kaggle version terminates abruptly, recover output/version logs; do not invent a checkpoint or restart an already-used session. Native epoch100 terminal is an unproven real-GPU path and must pass its separate terminal attestation when reached.

**No unsealing B-TEST, no model changes, no automatic official authorization.**


In [ ]:
from pathlib import Path
import hashlib, io, json, os, runpy, shutil, subprocess, sys, time, zipfile
from datetime import datetime, timezone

# --- REVIEWED IDENTITY. Never change these without new release review. ---
SHA = "fa6a7675daae385a04c7b77e69a9de29f9e1b2f3"
BASE = "b783eb3a470ab010785a8a24a90a666e9a3f2ed6"
PORTABLE_SHA = "3f0a386ace43453921ddd84b0e5f1aef5dacb72493d11fd5fbadcfad3c974e3c"
RESTAGER_SHA = "88cd4a88c373f451e54ba64736e961e735757d94b3779feea55a526868c45f4f"
EXECUTE_APPROVED_OFFICIAL = False  # NEVER change before separately recorded user consent
SESSION_LABEL = "S01"               # increment for every new Kaggle Save Version
RID = {"MV00":"C1OFFICIAL_MV00_20261022", "MV01":"C1OFFICIAL_MV01_20261022"}

# Each model has an INDEPENDENT cursor. Never infer one arm's epoch from the other.
# For RESUME: start = independently verified prior completed epoch + 1.
# For RESUME: resume_sha256 = independently verified ORIGINAL G4 .zipblob hash.
ARMS = {
    "MV00": {"mode":"FRESH", "start":1, "target":12, "resume_sha256":None},
    "MV01": {"mode":"FRESH", "start":1, "target":12, "resume_sha256":None},
}
SESSION_SECONDS = 32400     # 9-hour native-worker wall-clock deadline
MINIMUM_EPOCH_SECONDS = 1800
EXPORT_RESERVE_SECONDS = 3600  # > C1 required 30-minute minimum
ROOT = Path("/kaggle/input")
REPO = Path("/kaggle/working/c1_official_fixed_checkout")
PLAN = Path("/kaggle/working/CHAT27_C1_OFFICIAL_PLANS_" + SESSION_LABEL)
OUT  = Path("/kaggle/working/CHAT27_C1_OFFICIAL_OUTPUT_" + SESSION_LABEL)
LOG  = Path("/kaggle/working/CHAT27_C1_OFFICIAL_LOGS_" + SESSION_LABEL)

def need(ok, reason):
    if not ok: raise RuntimeError("CHAT27_OFFICIAL_PRECHECK_FAIL_CLOSED_" + reason)

def digest(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for b in iter(lambda:f.read(1<<20),b""): h.update(b)
    return h.hexdigest()

need(ROOT.is_dir(), "KAGGLE_INPUTS_MISSING")
need(SESSION_LABEL.isalnum() and 1<=len(SESSION_LABEL)<=16, "SESSION_LABEL")
need(set(ARMS)==set(RID) and RID["MV00"]!=RID["MV01"], "TWO_INDEPENDENT_ARMS")
for arm, a in ARMS.items():
    need(set(a)=={"mode","start","target","resume_sha256"}, "ARM_KEYS_"+arm)
    need(a["mode"] in ("FRESH","RESUME"), "MODE_"+arm)
    st, tar = a["start"],a["target"]
    need(type(st) is int and type(tar) is int and 1<=st<=tar<=100 and tar-st+1<=12,
         "SESSION_BOUNDS_"+arm)
    if a["mode"]=="FRESH":
        need(st==1 and a["resume_sha256"] is None, "FRESH_MUST_RESTART_EPOCH1_"+arm)
    else:
        v=a["resume_sha256"]
        need(st>=2 and type(v) is str and len(v)==64 and
             all(c in "0123456789abcdef" for c in v), "REQUIRE_INDEPENDENT_G4_SHA_"+arm)
need(not any(x.exists() or x.is_symlink() for x in (REPO,PLAN,OUT,LOG)),
     "SESSION_MUST_START_CLEAN")
need(type(EXECUTE_APPROVED_OFFICIAL) is bool and
     0<SESSION_SECONDS<=36000 and EXPORT_RESERVE_SECONDS>=1800, "SESSION_BUDGET")
gpu=subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True)
gpu_lines=[v for v in gpu.stdout.splitlines() if v.startswith("GPU ")]
need(gpu.returncode==0 and len(gpu_lines)==2 and
     all("T4" in x.upper() for x in gpu_lines), "TWO_PHYSICAL_T4_REQUIRED")
print("T4_X2_AND_SESSION_CONFIG_PASS",SESSION_LABEL,flush=True)
print("OFFICIAL_EXECUTION_FLAG",EXECUTE_APPROVED_OFFICIAL, "B_TEST_NONE", flush=True)


In [ ]:
# Exact Git checkout; R8 verifies additive-only release ancestry.
subprocess.run(["git","clone","--no-checkout",
                "https://github.com/nithoukhan1/ResEMA.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",SHA],check=True)
need(subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],
                            text=True).strip()==SHA, "CHECKOUT_SHA")
need(REPO.is_dir() and not REPO.is_symlink() and
     not any(p.is_symlink() for p in (REPO,REPO.parent)), "CHECKOUT_SYMLINK")
REL=REPO/"research/r9_training"
C1=REL/"official_candidate"
CODE=REL/"code"
sys.path[:0]=[str(C1),str(CODE)]
from R9_OFFICIAL_RELEASE_IDENTITY_R8 import validate_checkout
identity=validate_checkout(REPO,SHA)
need(identity["status"]=="R8_EXACT_CHECKOUT_SHA_AND_ADDITIVE_DIFF_VERIFIED_NOT_REMOTE_PROOF",
     "RELEASE_IDENTITY")
print("EXACT_FROZEN_C1_RELEASE_PASS",SHA,flush=True)


In [ ]:
# Restore the 16 historical working paths with the exact Chat25 corrected restager.
NAME="CHAT25_C1_R0_STALE_PATH_SAFE_RESTAGE_CORRECTED.py"
def member_safe(s):
    from pathlib import PurePosixPath
    p=PurePosixPath(s)
    return not p.is_absolute() and ".." not in p.parts and "\\" not in s
def find_inside_zip(blob,depth=0):
    if depth>3:return []
    found=[]
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for m in z.infolist():
            if m.is_dir() or not member_safe(m.filename):continue
            if (m.filename==NAME or m.filename.endswith("/"+NAME)):
                need(m.file_size<2_000_000, "RESTAGER_SIZE")
                found.append(z.read(m))
            elif m.filename.lower().endswith(".zip") and m.file_size<120_000_000:
                found.extend(find_inside_zip(z.read(m),depth+1))
    return found
restagers=[p.read_bytes() for p in ROOT.rglob(NAME) if p.is_file() and not p.is_symlink()]
if not restagers:
    for p in ROOT.rglob("*"):
        if (p.is_file() and not p.is_symlink() and
            "DETECTION_PROJECT_MASTER_MIGRATION_CHAT1_TO_CHAT25_FOR_CHAT26_2026-10-21_PORTABLE" in p.name and
            p.suffix.lower() in (".zip",".zipblob")):
            need(digest(p)==PORTABLE_SHA, "CHAT25_PORTABLE_WRONG_BYTES")
            restagers.extend(find_inside_zip(p.read_bytes()))
need(bool(restagers), "ORIGINAL_CORRECTED_R0_RESTAGER_NOT_MOUNTED")
need(len({hashlib.sha256(v).hexdigest() for v in restagers})==1,
     "CONFLICTING_RESTAGER_COPIES")
script=Path("/kaggle/working")/NAME
with script.open("xb") as f:f.write(restagers[0])
need(digest(script)==RESTAGER_SHA,"CORRECTED_R0_SHA_MISMATCH")
expected_config=Path("/kaggle/input/datasets/nithoukhan2/detect-1/R7D_RESTORE_CONTEXT_SUPPORT_R1/inputs/RECOVERY_CONFIG.json")
expected_sci=REL/"frozen_source/ultralytics/models/yolo/detect/d4e_r3_read_only_identity_preflight.py"
need(expected_config.is_file() and not expected_config.is_symlink() and
     expected_sci.is_file() and not expected_sci.is_symlink(),
     "R7D_CONFIG_OR_FROZEN_SCIENCE_NOT_MOUNTED")
runpy.run_path(str(script),run_name="__main__")
print("EXACT_CHAT25_R0_RESTAGE_EXECUTED",flush=True)


In [ ]:
# Identify the exact original six inputs by SHA, never by convenient filename alone.
PINS={
 "runtime_yaml":("f2cc6e31f5f1be152f4dd7603a397e142963923f1d35de613eb7c61b168bf0c0",(".yaml",".yml")),
 "assignment":("87aa5e9089c6282df14ae32917946d70d9f56eaa9a5d63443985877ab9dd1ef4",(".csv",".json")),
 "visibility":("9de034fa7df668d17effa843674c097341af9dd6461963f9e2c454c5f358b289",(".json",)),
 "pretrained_best":("620594d3560310b5d24243e1b321ce4463b0795b295891c47f6ec177e5272ac1",(".pt",)),
 "pretrained_archive":("4e3a811be5b8126e788ca4b7854e2df2b070a08e2abef396",(".bin",".zip",".zipblob"))
}
# Corrected full archive pin from Chat26 historical notebook and master handoff:
PINS["pretrained_archive"]=("4e3a811be5b8126e788ca4b7854e2df2b070a08e2abef396",(".bin",".zip",".zipblob"))
MOUNTED=("d4e-r12-accepted-assets","d4e-r12-raw-archives-v1",
         "detect-1","detection","d4f-r7d-legacy-exact-source-evidence-r1")
files=[]
for dirpath, dirs, fnames in os.walk(ROOT):
    dirs[:]=[d for d in dirs if not d.startswith(".")]
    for name in fnames:
        p=Path(dirpath)/name
        if any(s in str(p) for s in MOUNTED) and p.is_file() and not p.is_symlink():
            files.append(p)
INPUT={}
for key, (pin,suffixes) in PINS.items():
    matches=[p for p in files if p.name.lower().endswith(suffixes) and digest(p)==pin]
    need(bool(matches),"MISSING_EXACT_PIN_"+key)
    INPUT[key]=str(sorted(matches,key=str)[0])
configs=[p for p in files if p.name=="RECOVERY_CONFIG.json" and
         "R7D_RESTORE_CONTEXT_SUPPORT_R1/inputs" in str(p)]
need(len(configs)==1 and configs[0]==expected_config, "ORIGINAL_R7D_CONFIG")
INPUT["config_path"]=str(configs[0])
print("ORIGINAL_SIX_INPUT_IDENTITIES_PASS",json.dumps(INPUT,indent=2),flush=True)


In [ ]:
# Only verified, genuinely offsite official per-arm G4 can enable RESUME.
from R9_OFFSESSION_TRANSPORT_R1 import inspected_archive
def prior_triplet(arm, settings):
    if settings["mode"]=="FRESH":return (None,None,None)
    ep=settings["start"]-1
    pin=settings["resume_sha256"]
    blobs=[p for p in ROOT.rglob(f"{arm}_epoch{ep:03d}_G4_LOCAL_EXPORT.zip.zipblob")
           if p.is_file() and not p.is_symlink() and digest(p)==pin]
    need(len(blobs)==1,"REQUIRE_ONE_EXACT_VERIFIED_OFFSITE_G4_"+arm)
    blob=blobs[0]
    side=Path(str(blob)+".sha256")
    need(side.is_file() and side.read_text().split()==[pin,blob.name],
         "ARCHIVE_SIDE_CAR_"+arm)
    checked=inspected_archive(blob,sidecar=side)
    need(checked["status"]=="PASS_BYTE_LEVEL_ARCHIVE_VERIFICATION_NOT_GPU_REPRODUCTION"
         and checked["arm"]==arm and checked["completed_epoch"]==ep-1
         and checked["terminal"] is False and checked["zip_sha256"]==pin,
         "ARCHIVE_NOT_MATCHING_CURSOR_"+arm)
    found=[]
    for f in blob.parent.rglob("*.json"):
        if not f.is_file() or f.is_symlink():continue
        try: j=json.loads(f.read_text("utf-8"))
        except (ValueError, OSError, UnicodeError):continue
        if (j.get("schema")=="R9_OFFSITE_CHECK_NOT_NATIVE_RESUME_ACCEPTANCE" and
            j.get("arm")==arm and j.get("completed_epoch")==ep-1 and
            j.get("terminal") is False and j.get("zip_sha256")==pin):
            found.append(f)
    need(len(found)==1, "REQUIRE_INDEPENDENT_WINDOWS_OFFSESSION_RECEIPT_"+arm)
    return (str(blob),str(side),str(found[0]))
RESUME={arm:prior_triplet(arm,ARMS[arm]) for arm in RID}
print("PER_ARM_OFFSITE_RESUME_INPUTS",{
    a:("FRESH" if RESUME[a][0] is None else "OFFSITE_G4_BYTES_VERIFIED") for a in RID},
    flush=True)


In [ ]:
# Source/data/R0 pin verification and official 2-arm session plan; still no GPU.
from R9_PRODUCTION_PLAN_FACTORY_C1 import create
from R9_PRODUCTION_APPROVAL_GATE_C1 import validate as verify_approval
requests={}
for arm in RID:
    a=ARMS[arm]
    triple=RESUME[arm]
    requests[arm]={"mode":"FRESH_FROM_EARLY" if a["mode"]=="FRESH" else "RESUME_FROM_OUR_OWN_RUN",
                   "start_epoch_one_based":a["start"],"target_epoch_one_based":a["target"],
                   "resume_archive":triple[0],"resume_sidecar":triple[1],
                   "resume_external_attestation":triple[2]}
request={"run_ids":RID, "arms":requests, "monotonic_session_seconds":SESSION_SECONDS,
         "minimum_epoch_seconds":MINIMUM_EPOCH_SECONDS,
         "reserve_export_seconds":EXPORT_RESERVE_SECONDS}
result=create(INPUT,request,checkout=str(REPO),release_sha=SHA,output_dir=str(PLAN))
need(result["status"]=="TWO_ARM_PINNED_PLANS_CREATED_NO_GPU","PLAN_FACTORY")
specs={arm:json.loads((PLAN/(arm+".json")).read_text()) for arm in RID}
review=verify_approval(specs,None,SHA,execute=False)
need(review["official_training_authorized"] is False, "NO_IMPLICIT_AUTH")
print("TWO_ARM_OFFICIAL_PLAN_REVIEW_PASS_NO_GPU",json.dumps(result,indent=2),flush=True)


In [ ]:
# Actual execution stays LOCKED until a private, explicit user approval record is mounted.
# This notebook NEVER writes an authorization file itself.
APPROVAL_NAME="CHAT27_C1_OFFICIAL_USER_AUTHORIZATION.json"
if not EXECUTE_APPROVED_OFFICIAL:
    print("REVIEW_ONLY_NO_GPU: exact release/source/data/plans validated.")
    print("No training was launched. Explicit final approval and private authorization JSON required.")
else:
    candidates=[p for p in ROOT.rglob(APPROVAL_NAME)
                if p.is_file() and not p.is_symlink()]
    need(len(candidates)==1,"ONE_PRIVATE_USER_AUTHORIZATION_FILE_REQUIRED")
    AUTH=candidates[0]
    approval=json.loads(AUTH.read_text("utf-8"))
    auth=verify_approval(specs,approval,SHA,execute=True)
    need(auth["status"]=="OFFICIAL_RUN_APPROVAL_RECORD_MATCHED_NOT_EXECUTION_PROOF",
         "EXACT_USER_AUTHORITY_AND_RUN_IDENTITIES")
    import torch
    need(torch.cuda.is_available() and torch.cuda.device_count()==2 and
         all("T4" in torch.cuda.get_device_name(i).upper() for i in (0,1)),
         "CUDA_T4_X2_REQUIRED")
    need(not OUT.exists() and not LOG.exists(),"FRESH_OUTPUT_LOG_ROOTS_REQUIRED")
    need(all(not Path(specs[a]["working_root"]).exists() for a in RID),
         "WORKER_OUTPUT_MUST_BE_FRESH")
    cmd=[sys.executable,"-u",str(C1/"R9_PRODUCTION_DUAL_SESSION_C1.py"),
         "--mv00",str(PLAN/"MV00.json"),"--mv01",str(PLAN/"MV01.json"),
         "--release-sha",SHA,"--authorization",str(AUTH),
         "--output-root",str(OUT),"--logs-root",str(LOG),
         "--execute-approved-final-run"]
    supervisor=Path("/kaggle/working")/("CHAT27_C1_"+SESSION_LABEL+"_SUPERVISOR.log")
    incident=Path("/kaggle/working")/("CHAT27_C1_"+SESSION_LABEL+"_INCIDENT.zip")
    need(not supervisor.exists() and not incident.exists(), "ORIGINAL_LOGS_MUST_BE_FRESH")

    def tail(path):
        if not path.is_file():return "not-yet-created"
        with path.open("rb") as f:
            f.seek(max(0,path.stat().st_size-8192))
            b=f.read().decode("utf-8",errors="replace")
        a=[v.strip() for v in b.replace("\r","\n").splitlines() if v.strip()]
        return a[-1][:230] if a else "empty"

    def incident_archive():
        if incident.exists():return
        picks=[PLAN/"MV00.json",PLAN/"MV01.json",PLAN/"PLAN_REVIEW.json",supervisor,
               LOG/"MV00_native_stdout_stderr.log",LOG/"MV01_native_stdout_stderr.log",
               OUT/"C1_OUTPUT_INDEX.json"]
        with zipfile.ZipFile(incident,"x",zipfile.ZIP_DEFLATED) as z:
            for p in picks:
                if p.is_file() and not p.is_symlink():
                    z.write(p,p.relative_to("/kaggle/working").as_posix())
        Path(str(incident)+".sha256").write_text(
            digest(incident)+"  "+incident.name+"\n",encoding="utf-8")
        print("PRESERVED_NONCANONICAL_INCIDENT",str(incident),flush=True)

    print("OFFICIAL_DUAL_T4_SESSION_START",SESSION_LABEL,flush=True)
    print("TRAINING_LABEL_SCAN_CAN_TAKE_MANY_MINUTES; NO INTERRUPT_OR_RERUN",flush=True)
    with supervisor.open("x",encoding="utf-8") as sink:
        proc=subprocess.Popen(cmd,stdout=sink,stderr=subprocess.STDOUT,
                              stdin=subprocess.DEVNULL)
        try:
            while proc.poll() is None:
                print("UTC_HEARTBEAT",datetime.now(timezone.utc).isoformat(),flush=True)
                for a in RID:
                    print(a,tail(LOG/(a+"_native_stdout_stderr.log")),flush=True)
                time.sleep(75)
        except KeyboardInterrupt:
            print("INTERRUPTED_DO_NOT_RESTART_IN_THIS_KAGGLE_SESSION",flush=True)
            if proc.poll() is None:
                proc.terminate()
                try:proc.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    proc.kill();proc.wait()
            incident_archive()
            raise
        rc=proc.wait()
    print("OFFICIAL_DUAL_SUPERVISOR_EXIT",rc,flush=True)
    if rc!=0:
        incident_archive()
        print("SUPERVISOR_TAIL",tail(supervisor),flush=True)
        for a in RID:
            print("ARM_LOG_TAIL",a,tail(LOG/(a+"_native_stdout_stderr.log")),flush=True)
        raise RuntimeError("OFFICIAL_SESSION_FAILED_KEEP_ALL_ORIGINAL_ARTIFACTS_NO_AUTO_RETRY")
    index_path=OUT/"C1_OUTPUT_INDEX.json"
    if not index_path.is_file():
        incident_archive()
        raise RuntimeError("OFFICIAL_OUTPUT_INDEX_MISSING")
    idx=json.loads(index_path.read_text("utf-8"))
    if not (idx.get("both_arms_completed") is True and
            idx.get("status")=="BOTH_ARMS_LOCAL_OUTPUTS_VERIFIED_EXTERNAL_PENDING" and
            idx.get("run_class")=="OFFICIAL_100EPOCH" and
            idx.get("published_release_commit")==SHA and
            idx.get("b_test_access")=="NONE" and
            idx.get("offsite_persistence_verified") is False and
            not idx.get("failures") and set(idx.get("arms",{}))==set(RID)):
        incident_archive()
        raise RuntimeError("OFFICIAL_DUAL_ARM_OUTPUT_STATUS_INVALID")
    summary={}
    for arm, e in idx["arms"].items():
        cursor=e["completed_epoch_zero_based"]
        need(e["run_id"]==RID[arm] and
             ARMS[arm]["start"]-1<=cursor<ARMS[arm]["target"] and
             e["terminal"] is (cursor==99), "ARM_CURSOR_OR_TERMINAL_"+arm)
        blob=OUT/e["zip_relative"]
        need(blob.is_file() and blob.name.endswith(".zipblob") and
             digest(blob)==e["zip_sha256"], "OUTPUT_ZIPBLOB_HASH_"+arm)
        side=Path(str(blob)+".sha256")
        with side.open("x",encoding="utf-8") as f:
            f.write(e["zip_sha256"]+"  "+blob.name+"\n")
        worker_log=LOG/(arm+"_native_stdout_stderr.log")
        if worker_log.is_file():
            shutil.copy2(worker_log,OUT/(arm+"_native_stdout_stderr.log"))
        summary[arm]={"completed_epoch_one_based":cursor+1,
                      "terminal":e["terminal"],"g4_sha256":e["zip_sha256"],
                      "next_epoch_one_based":None if cursor==99 else cursor+2}
    shutil.copy2(supervisor,OUT/("CHAT27_C1_"+SESSION_LABEL+"_SUPERVISOR.log"))
    print("OFFICIAL_SESSION_LOCAL_G4_PASS_EXTERNAL_CHECK_REQUIRED",
          json.dumps(summary,indent=2),flush=True)
    print("DO_NOT_RESUME_UNTIL_INDEPENDENT_WINDOWS_OFFSITE_CHECK_PASSES",flush=True)


## After each official Kaggle session
1. Save the version output, download original arm `G4_LOCAL_EXPORT.zip.zipblob`, its `.zipblob.sha256` sidecar, `C1_OUTPUT_INDEX.json`, and original worker and supervisor logs.
2. Independently verify original `.zipblob` on Windows against its SHA and original G4 manifest using the frozen `R9_OFFSESSION_TRANSPORT_R1.py` verifier, and produce **separate** external attestation JSON for each arm.
3. Preserve original bytes without cleanup. Re-upload independently verified artifacts as a private Kaggle dataset; keep the same **official run IDs** and **release SHA**.
4. Update the next notebook version independently for MV00 and MV01: set `mode="RESUME"`, each `start=prior_completed+1`, `target=min(start+11,100)`, each exact original external `resume_sha256`, and increment `SESSION_LABEL`.
5. Only original terminal epoch100 G4 plus final-eval and external SHA verification establishes completed official training. Never use nonofficial Chat26 capsules as an official resume source. B-TEST stays sealed.

**Failure rules:** a missing, incomplete, corrupted, locally unverified, or unverified-offsite checkpoint is a BLOCKER; do not infer progress from best.pt SHA or the training console alone. One model failing does not justify silently declaring both complete. Do not restart training in the same Kaggle session after partial failure.
